In [ ]:
# CELL 1 — CONFIGURATION, FILE DISCOVERY AND SQLITE SQL ENGINE
from pathlib import Path
import os, re, sqlite3, zipfile, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
OUT = ROOT / 'Asheville_Project9_Outputs'
OUT.mkdir(parents=True, exist_ok=True)
# If needed, explicitly supply full paths to the three files below.
LISTINGS_FILE = None
REVIEWS_FILE = None
NEIGHBOURHOODS_FILE = None

def find_dataset(name, override=None):
    if override:
        p=Path(override)
        if not p.is_file():raise FileNotFoundError(f'Missing {name}: {p}')
        return p
    search_dirs=[ROOT,ROOT/'data',ROOT/'datasets',Path.cwd()]
    matches=[]
    for d in search_dirs:
        if d.is_dir():matches += [x for x in d.glob('*.csv') if x.name.lower()==name]
    matches=list(dict.fromkeys(matches))
    if not matches:raise FileNotFoundError(f'{name} not found. Upload it to Colab Files or set its path in Cell 1.')
    if len(matches)>1:print(f'Multiple {name} files found; using {matches[0]}')
    return matches[0]

LISTINGS_FILE=find_dataset('listings.csv',LISTINGS_FILE)
REVIEWS_FILE=find_dataset('reviews.csv',REVIEWS_FILE)
try:NEIGHBOURHOODS_FILE=find_dataset('neighbourhoods.csv',NEIGHBOURHOODS_FILE)
except FileNotFoundError:NEIGHBOURHOODS_FILE=None;print('WARNING: neighbourhoods.csv absent; geographic crosswalk will be omitted.')
print('Listings:',LISTINGS_FILE,'\nReviews:',REVIEWS_FILE,'\nNeighborhoods:',NEIGHBOURHOODS_FILE)
DB=OUT/'asheville_analysis.sqlite'
if DB.exists():DB.unlink()  # Fresh run prevents stale rows
con=sqlite3.connect(str(DB))
print('SQL ENGINE: SQLite',sqlite3.sqlite_version)

In [ ]:
# CELL 2 — INSPECT, CLEAN AND INGEST ALL THREE FILES
# Read listings and neighbourhoods as text to retain IDs/ZIP labels and diagnose missingness.
def normalize_cols(df):
    df=df.copy()
    df.columns=[re.sub(r'[^a-z0-9]+','_',str(c).lower().strip()).strip('_') for c in df.columns]
    return df.loc[:,~df.columns.duplicated()]
def source_file(path):
    for enc in ('utf-8-sig','utf-8','latin-1'):
        try:return pd.read_csv(path,dtype=str,low_memory=False,encoding=enc,on_bad_lines='warn')
        except UnicodeDecodeError:pass
    raise ValueError(f'Cannot decode {path}')

listings=normalize_cols(source_file(LISTINGS_FILE))
need={'id','host_id','room_type','price'}
assert not (need-set(listings.columns)), f'Missing essential listing columns: {need-set(listings.columns)}'
if 'neighbourhood' not in listings.columns:
    if 'neighbourhood_cleansed' in listings.columns:listings['neighbourhood']=listings['neighbourhood_cleansed']
    else:listings['neighbourhood']='UNKNOWN'
for c in ['neighbourhood_group','host_is_superhost','calculated_host_listings_count','minimum_nights','reviews_per_month','availability_365']:
    if c not in listings.columns:listings[c]=np.nan
listings=listings.replace(r'^\s*$',np.nan,regex=True)
listings['id']=listings['id'].astype('string').str.strip()
listings['host_id']=listings['host_id'].astype('string').str.strip()
for c in ['neighbourhood','room_type']:
    listings[c]=listings[c].fillna('UNKNOWN').astype(str).str.strip().str.upper()
listings['neighbourhood_group']=listings['neighbourhood_group'].fillna('UNKNOWN').astype(str).str.strip().str.upper()
listings['superhost']=listings['host_is_superhost'].fillna('').str.lower().map({'t':'YES','true':'YES','yes':'YES','f':'NO','false':'NO','no':'NO'}).fillna('UNKNOWN')
listings['price_usd']=pd.to_numeric(listings['price'].astype(str).str.replace(r'[$,]','',regex=True),errors='coerce')
for c in ['minimum_nights','calculated_host_listings_count','reviews_per_month','availability_365']:
    listings[c]=pd.to_numeric(listings[c],errors='coerce')
quality=pd.DataFrame([{'source_listings':len(listings),'missing_listing_ids':int(listings.id.isna().sum()),'duplicate_listing_id_rows':int(listings.id.duplicated().sum()),'missing_neighbourhood':int((listings.neighbourhood=='UNKNOWN').sum()),'missing_neighbourhood_group':int((listings.neighbourhood_group=='UNKNOWN').sum()),'missing_or_invalid_price':int(listings.price_usd.isna().sum()),'nonpositive_price':int((listings.price_usd<=0).sum()),'extreme_price_over_1000':int((listings.price_usd>1000).sum()),'missing_minimum_nights':int(listings.minimum_nights.isna().sum()),'missing_superhost_status':int((listings.superhost=='UNKNOWN').sum())}])
quality.to_csv(OUT/'data_quality_audit.csv',index=False)
assert listings.id.notna().any(), 'No valid listing IDs'
# A listing is counted once; duplicate IDs are audited, not counted multiple times.
listing_dupes=listings[listings.id.duplicated(keep=False)].copy()
listing_dupes[['id','host_id','price','neighbourhood']].to_csv(OUT/'duplicate_listing_ids_audit.csv',index=False)
listings=listings.dropna(subset=['id']).drop_duplicates('id',keep='first').copy()
listings['valid_price']=listings.price_usd.where((listings.price_usd>0)&(listings.price_usd<=10000))
# Price > $1,000 is flagged for review, not silently discarded; >$10,000 excluded from central-price metrics.
base=['id','host_id','neighbourhood','neighbourhood_group','room_type','superhost','price_usd','valid_price','minimum_nights','calculated_host_listings_count','reviews_per_month','availability_365']
listings[base].to_sql('listings',con,index=False,if_exists='replace')
con.execute('CREATE UNIQUE INDEX idx_listing_id ON listings(id)')

# Ingest reviews in chunks: SQLite is the central analysis database, not a giant pandas merge.
review_head=normalize_cols(pd.read_csv(REVIEWS_FILE,nrows=0))
assert {'listing_id','date'}.issubset(review_head.columns),f'reviews.csv needs listing_id/date; got {review_head.columns.tolist()}'
con.execute('CREATE TABLE reviews (listing_id TEXT, review_date TEXT, review_year INTEGER, review_month TEXT)')
read_reviews=0;valid_reviews=0;invalid_dates=0
for chunk in pd.read_csv(REVIEWS_FILE,dtype=str,chunksize=100000,low_memory=False,on_bad_lines='warn'):
    chunk=normalize_cols(chunk)
    dt=pd.to_datetime(chunk.date,errors='coerce')
    ids=chunk.listing_id.astype('string').str.strip()
    clean=pd.DataFrame({'listing_id':ids,'review_date':dt.dt.strftime('%Y-%m-%d'),'review_year':dt.dt.year,'review_month':dt.dt.strftime('%Y-%m')})
    read_reviews+=len(clean);valid_reviews+=int(clean.review_date.notna().sum());invalid_dates+=int(clean.review_date.isna().sum())
    clean.to_sql('reviews',con,if_exists='append',index=False)
con.commit()
con.execute('CREATE INDEX idx_review_listing ON reviews(listing_id)')
con.execute('CREATE INDEX idx_review_year ON reviews(review_year)')
con.execute('CREATE INDEX idx_review_month ON reviews(review_month)')

if NEIGHBOURHOODS_FILE:
    neighborhoods=normalize_cols(source_file(NEIGHBOURHOODS_FILE)).replace(r'^\s*$',np.nan,regex=True)
    neighborhoods.to_sql('neighbourhoods_reference',con,index=False,if_exists='replace')
    neighborhoods[['neighbourhood']].drop_duplicates().to_csv(OUT/'neighbourhood_reference.csv',index=False) if 'neighbourhood' in neighborhoods else None
    print('Neighborhood reference rows:',len(neighborhoods),'fields:',neighborhoods.columns.tolist())
else:print('Neighborhood reference: not supplied.')
pd.DataFrame([{'reviews_source_rows':read_reviews,'valid_review_dates':valid_reviews,'invalid_review_dates':invalid_dates}]).to_csv(OUT/'reviews_quality_audit.csv',index=False)
print('LISTING AUDIT');display(quality)
print(f'Loaded {len(listings):,} unique listings and {read_reviews:,} review records into SQLite.')

In [ ]:
# CELL 3 — SQL LISTINGS + REVIEWS JOINS, PRICE OUTLIERS, QUALITY RECONCILIATION
# Aggregate reviews first to avoid inflating listing counts in downstream joins.
con.executescript('''DROP VIEW IF EXISTS listing_enriched;
DROP VIEW IF EXISTS review_counts;
CREATE VIEW review_counts AS
 SELECT listing_id, COUNT(*) AS review_rows,
 SUM(CASE WHEN review_date IS NOT NULL THEN 1 ELSE 0 END) AS dated_reviews,
 MIN(review_date) AS first_review,MAX(review_date) AS last_review
 FROM reviews GROUP BY listing_id;
CREATE VIEW listing_enriched AS
 SELECT l.*, COALESCE(r.review_rows,0) AS review_count_file,
 COALESCE(r.dated_reviews,0) AS dated_reviews,
 r.first_review,r.last_review
 FROM listings l LEFT JOIN review_counts r ON l.id=r.listing_id;
''')
summary=pd.read_sql_query("""SELECT COUNT(*) unique_listings,COUNT(valid_price) priced_listings,
 ROUND(AVG(valid_price),2) avg_price_usd,
 SUM(review_count_file) linked_review_rows,
 SUM(CASE WHEN review_count_file>0 THEN 1 ELSE 0 END) listings_with_reviews
 FROM listing_enriched""",con)
prices=pd.read_sql_query('SELECT valid_price FROM listing_enriched WHERE valid_price IS NOT NULL ORDER BY valid_price',con).valid_price
summary['median_price_usd']=prices.median() if not prices.empty else np.nan
summary['p95_price_usd']=prices.quantile(.95) if not prices.empty else np.nan
summary.to_csv(OUT/'market_overview.csv',index=False)
# Unmatched review IDs may reflect listings that have left the snapshot; retain their records in review time trends.
unmatched=pd.read_sql_query("""SELECT COUNT(*) AS review_rows_without_current_listing
 FROM reviews r LEFT JOIN listings l ON r.listing_id=l.id WHERE l.id IS NULL""",con)
unmatched.to_csv(OUT/'unmatched_reviews_audit.csv',index=False)
assert int(pd.read_sql_query('SELECT COUNT(*) n FROM listing_enriched',con).n.iloc[0])==len(listings)
assert int(summary.linked_review_rows.iloc[0])+int(unmatched.iloc[0,0])==read_reviews
print('MARKET OVERVIEW');display(summary)
print('UNMATCHED HISTORICAL REVIEWS');display(unmatched)
print('Note: review counts are a proxy for guest activity, not booked nights, occupancy, or revenue.')

In [ ]:
# CELL 4 — SQL PRICING, ROOM TYPES, NEIGHBORHOOD AND MINIMUM NIGHTS
def export(name,sql,params=()):
    data=pd.read_sql_query(sql,con,params=params)
    data.to_csv(OUT/(name+'.csv'),index=False)
    return data
room=export('room_type_distribution','''SELECT room_type,COUNT(*) listing_count,
 ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM listings),2) pct_listings,
 ROUND(AVG(valid_price),2) avg_nightly_price
 FROM listing_enriched GROUP BY room_type ORDER BY listing_count DESC''')
hood=export('neighbourhood_pricing','''SELECT neighbourhood,COUNT(*) listing_count,
 COUNT(valid_price) priced_count,ROUND(AVG(valid_price),2) avg_nightly_price,
 ROUND(AVG(CASE WHEN valid_price IS NOT NULL THEN review_count_file END),2) avg_reviews_per_priced_listing,
 SUM(review_count_file) linked_review_rows
 FROM listing_enriched GROUP BY neighbourhood ORDER BY listing_count DESC''')
# Small samples are excluded from "most expensive" rankings to avoid unstable outliers.
MIN_NEIGHBORHOOD_SAMPLE=10
top5=hood[(hood.priced_count>=MIN_NEIGHBORHOOD_SAMPLE)&(hood.neighbourhood!='UNKNOWN')].sort_values('avg_nightly_price',ascending=False).head(5)
top5.to_csv(OUT/'top5_expensive_neighbourhoods.csv',index=False)
nights=export('minimum_nights_pricing','''SELECT CASE WHEN minimum_nights IS NULL THEN 'UNKNOWN'
 WHEN minimum_nights=1 THEN '1 night' WHEN minimum_nights BETWEEN 2 AND 3 THEN '2–3 nights'
 WHEN minimum_nights BETWEEN 4 AND 7 THEN '4–7 nights'
 WHEN minimum_nights BETWEEN 8 AND 30 THEN '8–30 nights'
 ELSE '31+ nights' END AS min_nights_band,
 COUNT(*) listings,ROUND(AVG(valid_price),2) avg_price_usd
 FROM listing_enriched GROUP BY 1 ORDER BY listings DESC''')
price_band=export('price_distribution','''SELECT CASE WHEN valid_price IS NULL THEN 'INVALID/MISSING'
 WHEN valid_price<75 THEN 'Under $75' WHEN valid_price<150 THEN '$75–149'
 WHEN valid_price<250 THEN '$150–249' WHEN valid_price<400 THEN '$250–399'
 ELSE '$400+' END price_band,COUNT(*) listings
 FROM listing_enriched GROUP BY 1 ORDER BY listings DESC''')
print('ROOM TYPES');display(room)
print('TOP 5 EXPENSIVE NEIGHBORHOODS (>=10 VALID PRICES)');display(top5)
print('MINIMUM NIGHT STAY PRICE COMPARISON');display(nights)

In [ ]:
# CELL 5 — SQL REVIEW TRENDS, DEMAND PROXIES, HOST COMPETITION
# Review volume != bookings: review propensity, tenure and listing availability change counts.
reviews_year=export('reviews_by_year','''SELECT review_year year,COUNT(*) reviews
 FROM reviews WHERE review_year IS NOT NULL AND review_year>=2000
 GROUP BY review_year ORDER BY review_year''')
reviews_month=export('reviews_by_month','''SELECT review_month month,COUNT(*) reviews
 FROM reviews WHERE review_month IS NOT NULL AND review_year>=2000
 GROUP BY review_month ORDER BY review_month''')
review_season=export('review_calendar_seasonality','''SELECT SUBSTR(review_month,6,2) month_number,COUNT(*) reviews
 FROM reviews WHERE review_month IS NOT NULL AND review_year>=2000
 GROUP BY 1 ORDER BY 1''')
hood_demand=export('neighbourhood_review_activity','''SELECT neighbourhood,COUNT(*) unique_listings,
 SUM(review_count_file) linked_review_rows,
 ROUND(AVG(review_count_file),2) avg_reviews_per_listing,
 ROUND(AVG(reviews_per_month),2) avg_listings_reported_reviews_per_month,
 SUM(CASE WHEN review_count_file>0 THEN 1 ELSE 0 END) listings_with_reviews
 FROM listing_enriched GROUP BY neighbourhood ORDER BY linked_review_rows DESC''')
hosts=export('host_activity','''SELECT host_id,COUNT(*) actual_listing_count,
 MAX(calculated_host_listings_count) published_host_listing_count,
 ROUND(AVG(valid_price),2) avg_price_usd,
 SUM(review_count_file) linked_review_rows,
 MAX(superhost) superhost_label
 FROM listing_enriched WHERE host_id IS NOT NULL GROUP BY host_id
 ORDER BY actual_listing_count DESC,linked_review_rows DESC LIMIT 50''')
superhosts=export('superhost_comparison','''SELECT superhost,COUNT(*) listings,ROUND(AVG(valid_price),2) avg_price_usd,
 ROUND(AVG(review_count_file),2) avg_reviews_per_listing,
 ROUND(AVG(reviews_per_month),2) avg_reported_reviews_per_month
 FROM listing_enriched GROUP BY superhost ORDER BY listings DESC''')
host_size=export('host_size_pricing','''WITH host_counts AS (SELECT host_id,COUNT(*) host_listing_count FROM listings WHERE host_id IS NOT NULL GROUP BY host_id)
 SELECT CASE WHEN h.host_listing_count IS NULL THEN 'HOST UNKNOWN'
 WHEN h.host_listing_count=1 THEN '1 listing' WHEN h.host_listing_count BETWEEN 2 AND 4 THEN '2–4 listings'
 ELSE '5+ listings' END host_size,COUNT(*) listings,
 ROUND(AVG(l.valid_price),2) avg_price_usd,ROUND(AVG(l.review_count_file),2) avg_reviews_per_listing
 FROM listing_enriched l LEFT JOIN host_counts h ON l.host_id=h.host_id
 GROUP BY 1 ORDER BY listings DESC''')
assert int(room.listing_count.sum())==int(hood.listing_count.sum())==len(listings)
assert int(reviews_year.reviews.sum())<=valid_reviews
print('VALIDATION: PASS — room/neighborhood counts reconcile; reviews filtered to plausible years.')
print('REVIEW ACTIVITY BY NEIGHBORHOOD');display(hood_demand.head(10))
print('HOST SIZE');display(host_size)

In [ ]:
# CELL 6 — EXECUTIVE DASHBOARD + REVIEW COVERAGE WARNINGS
fig,axes=plt.subplots(2,2,figsize=(15,10))
r=room.head(6).iloc[::-1]
axes[0,0].barh(r.room_type,r.listing_count);axes[0,0].set_title('Listing inventory by room type')
pr=hood[(hood.priced_count>=10)&(hood.neighbourhood!='UNKNOWN')].nlargest(10,'avg_nightly_price').sort_values('avg_nightly_price')
axes[0,1].barh(pr.neighbourhood,pr.avg_nightly_price);axes[0,1].set_title('Nightly prices by neighborhood (min 10 listings)')
trend=reviews_month.tail(48)
axes[1,0].plot(trend.month,trend.reviews);axes[1,0].set_xticks(range(0,len(trend),max(1,len(trend)//8)));axes[1,0].tick_params(axis='x',rotation=40);axes[1,0].set_title('Review counts in latest 48 recorded months')
h=hood_demand.head(10).iloc[::-1]
axes[1,1].barh(h.neighbourhood,h.linked_review_rows);axes[1,1].set_title('Reviews attached to current listings by neighborhood')
for ax in axes.flat:ax.grid(alpha=.2);ax.set_axisbelow(True)
fig.suptitle('Asheville Airbnb — SQL Market Research Dashboard',size=16)
fig.tight_layout(rect=[0,0,1,.96]);fig.savefig(OUT/'asheville_airbnb_market_dashboard.png',dpi=150,bbox_inches='tight');plt.show()
print('Dashboard saved. NOTE: reviews are only an activity proxy; historical years/months may be partial.')

In [ ]:
# CELL 7 — EVIDENCE-BASED README, RECOMMENDATIONS, DEPENDENCIES AND PACKAGE
from zipfile import ZipFile, ZIP_DEFLATED

def mdtable(df,cols,n=10):
    if df.empty:return '_No records meet the comparison threshold._'
    d=df[cols].head(n).copy()
    for c in d.columns:
        if pd.api.types.is_numeric_dtype(d[c]):d[c]=d[c].round(2)
    rows=['| '+' | '.join(map(str,r))+' |' for r in d.itertuples(index=False,name=None)]
    return '| '+' | '.join(cols)+' |\n| '+' | '.join(['---']*len(cols))+' |\n'+'\n'.join(rows)

n=len(listings);mean=summary.avg_price_usd.iloc[0];median=summary.median_price_usd.iloc[0]
peak_year=(str(reviews_year.loc[reviews_year.reviews.idxmax(),'year']) if len(reviews_year) else 'unavailable')
peak_month=(str(reviews_month.loc[reviews_month.reviews.idxmax(),'month']) if len(reviews_month) else 'unavailable')
leading=hood_demand.iloc[0].neighbourhood if len(hood_demand) else 'unavailable'
leading_room=room.iloc[0].room_type if len(room) else 'unavailable'
report=f"""# Asheville Airbnb Market Opportunity — SQL Analytics

![Executive dashboard](asheville_airbnb_market_dashboard.png)

## Business question
Evaluate price positioning, neighborhood supply, host concentration and historic review activity to assess the potential for travel-service expansion in Asheville, North Carolina.

## Source and scope
Inside Airbnb: `listings.csv` (snapshot inventory), `reviews.csv` (historical reviews) and `neighbourhoods.csv` (optional reference table). Current listing inventory is not the same as the population of every historical listing. CSVs are not bundled because they may be large or subject to redistribution restrictions; obtain the same extract and point Cell 1 to the files.

## Core findings
- **Unique listing snapshot:** {n:,} listings.
- **Mean advertised nightly price:** ${mean:,.2f}; **median:** ${median:,.2f} (valid advertised prices >0 and <=$10,000). Fees/taxes not included.
- **Largest room type:** {leading_room}.
- **Most linked historical review rows:** {leading} (only reviews matched to current listing IDs).
- **Highest recorded yearly review count:** {peak_year}; **highest recorded calendar month:** {peak_month}. Boundary periods may be partial and review volume is not booked nights.

## Room type distribution
{mdtable(room,['room_type','listing_count','pct_listings','avg_nightly_price'])}

## Neighborhood listing supply and pricing
{mdtable(hood,['neighbourhood','listing_count','avg_nightly_price','linked_review_rows'])}

## Top 5 most expensive neighborhoods
Requires at least {MIN_NEIGHBORHOOD_SAMPLE} listings with valid advertised nightly prices. Rankings are descriptive and do not adjust for property size or room type.
{mdtable(top5,['neighbourhood','priced_count','avg_nightly_price'],5)}

## Minimum-stay rules and pricing
{mdtable(nights,['min_nights_band','listings','avg_price_usd'])}

## Review activity as a demand proxy
{mdtable(reviews_year,['year','reviews'],12)}

Historic review count can rise because listing coverage, review habits and platform adoption change. Review timestamps cannot establish stays, booking conversion, paid occupancy, seasonality-adjusted demand or revenue without independent booking data. Source reviews not linked to a current listing remain counted in citywide review trends and are separately audited.

## Neighborhood review activity
{mdtable(hood_demand,['neighbourhood','unique_listings','linked_review_rows','avg_reviews_per_listing'])}

## Hosts and Superhosts
{mdtable(host_size,['host_size','listings','avg_price_usd','avg_reviews_per_listing'])}

{mdtable(superhosts,['superhost','listings','avg_price_usd','avg_reviews_per_listing'])}

The top 50 hosts are in `host_activity.csv`. Superhost comparison is omitted if the field is missing and all statuses become UNKNOWN. Listing size is calculated from the observed snapshot, while `calculated_host_listings_count` is retained separately; these measures need not agree.

## Data-quality and method
- `data_quality_audit.csv` audits missing IDs, neighborhood groups, prices, extreme prices and Superhost status.
- `reviews_quality_audit.csv` records missing/unparseable review dates; `unmatched_reviews_audit.csv` shows review rows belonging to listings absent from the current snapshot.
- SQLite tables `listings` and `reviews` are linked on listing ID, with aggregated review counts to avoid double-counting listing inventory.
- Outliers above $1,000 are flagged; missing, nonpositive and above-$10,000 prices are excluded from central-price comparisons. The threshold is a pragmatic descriptive rule, not a market-validated cutoff.
- `neighbourhood_group` can legitimately be empty in this extract. Use the `neighbourhood` field for neighborhood comparisons. ZIP and geographic boundary validation depend on reference coverage.
- No unsupported causal conclusion is made about hosts, price, review activity or regulatory viability.

## Limitations and decision
A positive expansion recommendation requires confirming occupancy/bookings, costs, guest acquisition channels, seasonality, legal eligibility and comparable competitor offerings. **This descriptive report alone does not establish investment viability.** The provided CSVs determine the report scope; sample size and coverage should be verified before a final business decision.

## Reproduce
1. Open `Asheville_Airbnb_SQL_Market_Analysis.ipynb` in Google Colab.
2. Upload `listings.csv`, `reviews.csv` and `neighbourhoods.csv` into `/content` (or set full paths in Cell 1).
3. Run Cells 1–7 in order. SQLite handles review ingestion in chunks and SQL joins.
4. Use the exported CSVs, `asheville_airbnb_market_dashboard.png`, this README and `RECOMMENDATIONS.md` for GitHub publication. Download the `.ipynb` separately.

## Technology
Python, pandas, SQLite SQL, Matplotlib, Google Colab. Excel-compatible CSVs are provided; no Power BI workbook is claimed.

## GitHub repository
`Asheville-Airbnb-Market-Opportunity-Analytics` — https://github.com/new
"""
(OUT/'README.md').write_text(report,encoding='utf-8')
recommend=f"""# Asheville Airbnb — Business Recommendations

These are evidence-informed **screening actions** based on the supplied listing snapshot and review history, not proof of occupancy or investment returns.

## 1. Pilot in high-review neighborhoods (High)
**Evidence:** {leading} leads linked historical review totals in the snapshot.
**Action:** Test a small targeted travel-services pilot and verify recent review growth, competitor density, prices and property availability.
**KPI:** Conversion rate and pilot revenue per acquired customer.

## 2. Segment offerings by room type (High)
**Evidence:** {leading_room} represents the largest published room category.
**Action:** Benchmark traveler price preferences and service needs within each room category, avoiding unfair direct comparisons between entire homes and private rooms.
**KPI:** Segment gross margin and repeat-purchase rate.

## 3. Price against comparable neighborhoods (High)
**Evidence:** The sample mean nightly price is ${mean:,.2f} and median is ${median:,.2f}; outliers can skew the mean.
**Action:** Build separate local price bands by room type, bedrooms (if supplied) and minimum-stay policy.
**KPI:** Conversion by price band; median competitor price gap.

## 4. Test seasonality rather than equating reviews with bookings (High)
**Evidence:** Review counts peaked in {peak_month} in the recorded data, but boundaries and review behavior vary.
**Action:** Compare complete years and obtain independent booking/occupancy data before seasonal capacity commitments.
**KPI:** Occupancy and booked-night share, not raw review counts.

## 5. Diversify partner-host exposure (Medium)
**Evidence:** `host_activity.csv` and `host_size_pricing.csv` show listing concentration across observed host IDs.
**Action:** Avoid reliance on one large host; recruit a range of verified, compliant partners.
**KPI:** Share of partner inventory supplied by the five largest host accounts.

## 6. Evaluate minimum-night constraints (Medium)
**Evidence:** `minimum_nights_pricing.csv` segments listings by required stay length.
**Action:** Target offerings to permitted stay durations and compare achievable booking windows by property type.
**KPI:** Eligible listing share and booked-night conversion by minimum-stay band.

## 7. Verify regulatory and operational feasibility (Critical)
**Evidence:** Listing and review records do not verify local permit compliance, net earnings or actual availability.
**Action:** Conduct current legal checks, supply-partner verification, cost modeling and user research prior to expansion.
**KPI:** Eligible partner count, unit contribution margin and compliance pass rate.

## Conclusion
Asheville is a **candidate for further validation, not a confirmed profitable expansion market**. Conduct a controlled pilot only after legal, occupancy and unit-economics checks.
"""
(OUT/'RECOMMENDATIONS.md').write_text(recommend,encoding='utf-8')
(OUT/'requirements.txt').write_text('pandas>=2.0,<4\nnumpy>=1.24,<3\nmatplotlib>=3.7,<4\n',encoding='utf-8')
(OUT/'.gitignore').write_text('*.sqlite\n*.db\n.ipynb_checkpoints/\n__pycache__/\n.DS_Store\n',encoding='utf-8')
required=['README.md','RECOMMENDATIONS.md','asheville_airbnb_market_dashboard.png','market_overview.csv','neighbourhood_pricing.csv','reviews_by_year.csv','host_activity.csv']
assert all((OUT/f).exists() for f in required), 'Required project file missing.'
package=ROOT/'Asheville_Project9_GitHub_Package.zip'
with ZipFile(package,'w',ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p.suffix.lower() in ('.csv','.png','.md','.txt') or p.name=='.gitignore':z.write(p,p.name)
with ZipFile(package) as z:
    assert z.testzip() is None
    assert all(f in z.namelist() for f in required)
    count=len(z.namelist())
print('VALIDATION: PASS')
print('ZIP INTEGRITY: PASS',count,'files')
print('PACKAGE READY:',package)
print('NOTEBOOK: Download .ipynb separately from Colab and upload alongside extracted files.')
if Path('/content').exists():
    from google.colab import files
    files.download(str(package))